# One random snapshot, and everything behind it

Draw a `(country, week)` the corpus can actually score, lay out **every byte of
evidence** that reaches the rating model, print the prompt verbatim, and — on a
flag — send it and print the whole answer back.

The other two notebooks are each pinned to one anchor somebody typed:
`country_rating_walkthrough.ipynb` is `PT` today, `historical_rating_walkthrough.ipynb`
is `TR` on 2018-11-05. This one picks for you, which is the difference between
checking the case you thought of and seeing the ones you did not.

Every cell calls a production function. Nothing here re-implements the pipeline,
and it runs **masked** — the regime production actually scores in, where the
model is never told which country it is looking at.

**What it costs.** One cheap digest call per article (~20) plus up to three
body-rewrite calls: roughly **$0.05** per run. Digests are cached on a hash of
their content, so re-running the *same* anchor is nearly free; a new random week
is not. The scoring call is another ~$0.02 and is behind `RUN_SCORER`, off by
default.

**What it writes.** The `llm_artifact` digest and rewrite caches, in whichever
database `RISK_DB_TARGET` names. Nothing else — no `upsert_snapshot`, no
`write_run`, no `risk_snapshot` row, no lint row. The dashboard does not move.

In [1]:
import datetime
import json
import logging
import os
import pathlib
import random
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

# Repo root = the folder holding backend/main.py, so this works from any cwd.
PROJECT_ROOT = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "backend" / "main.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / "backend" / ".env")
load_dotenv()

# force=True is not optional: Jupyter installs its own root handler, so a plain
# basicConfig() is silently a no-op and no pipeline logs ever appear.
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s",
                    stream=sys.stdout, force=True)
# httpx logs a line per request and per redirect - dozens of them per digest
# batch, none about risk.
logging.getLogger("httpx").setLevel(logging.WARNING)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

from backend.data_fetching import curated_loader
from backend.data_upsert import data_push, store
from backend.llm import client as ai_client
from backend.llm import constants as ai_constants
from backend.llm import digest_engine, gazetteer, langchain_llm, rewrite, usage
from backend.llm import payload as llm_payload
from backend.news_fetching import snapshot_select
from backend.util import config, constants, pipeline, provenance
from backend.util.pilot import score as pilot_score

print("environment")
for key in ("OPENAI_API_KEY", "RISK_DB_TARGET"):
    print(f"  {key:<18} {os.getenv(key) or 'MISSING'}"
          if key == "RISK_DB_TARGET" else
          f"  {key:<18} {'set' if os.getenv(key) else 'MISSING'}")

print("\nthe instrument")
print(f"  scoring model      {ai_client.scoring_model()}")
print(f"  digest model       {ai_client.digest_model()}")
print(f"  seed               {ai_client.SEED}")
print(f"  prompt_version     {provenance.prompt_version()}")
print(f"  payload_version    {provenance.payload_version()}")
print(f"  gazetteer_version  {gazetteer.GAZETTEER_VERSION}")
print(f"  sweep / rewrite    {rewrite.SWEEP_VERSION} / {rewrite.REWRITE_VERSION}")

environment
  OPENAI_API_KEY     set
  RISK_DB_TARGET     dev

the instrument
  scoring model      gpt-4o-2024-08-06
  digest model       gpt-4o-mini-2024-07-18
  seed               42
  prompt_version     v4.5-no-publisher
  payload_version    p2
  gazetteer_version  aa63700b
  sweep / rewrite    9f4aee55 / 7078d4f6


## The draw

The **only** cell that decides what is scored. Everything below reads `ISO2` and
`AS_OF` and never re-picks, because a notebook that is half one country and half
another is worse than either — it looks complete and isn't.

Countries are read out of the `article` table rather than listed here. Today that
is whatever the harvest has reached; when prod fills in the rest of the roster
this cell finds them without being edited, which is the point.

The availability test is `snapshot_select.select` itself — the same function the
backfill calls — rather than a row count standing in for it. A proxy can disagree
with what the pipeline would actually get; the selector cannot.

In [2]:
PIN          = None   # ("PT", "2019-06-03") pins the anchor; None draws one
SEED         = None   # an int makes the draw repeatable
MIN_ARTICLES = 8      # under this the week is too thin to be worth reading
SCORING_MODE = "masked"   # production. "named" is the diagnostic twin.

MASKED = SCORING_MODE.startswith("masked")

# `_rows` is the store's own read helper. Private, and used here the way the
# other two notebooks use `pipeline._process_country` and
# `langchain_llm._digests_to_json`: this is the production path, seen from
# inside.
COVERAGE = store._rows("""
    SELECT country_iso2, min(published_at)::date AS first,
           max(published_at)::date AS last, count(*)::int AS n
      FROM article GROUP BY 1 ORDER BY 1
""")

# `config.country_name` raises KeyError off COUNTRY_ROSTER rather than guessing,
# so a corpus country the roster does not know is reported rather than dropped
# quietly - it means the harvest and the run universe disagree, which is worth
# seeing.
known, unknown = [], []
for row in COVERAGE:
    try:
        row["name"] = config.country_name(row["country_iso2"])
    except KeyError:
        unknown.append(row["country_iso2"])
        continue
    # A full 30-day window has to fit behind the anchor, or the first snapshot
    # is scored on a window that was half over before the harvest began.
    row["from"] = row["first"] + datetime.timedelta(days=30)
    if row["from"] <= row["last"]:
        known.append(row)

print(f"{len(known)} country/ies with a corpus:")
display(pd.DataFrame(known)[["country_iso2", "name", "first", "last", "n"]]
        .set_index("country_iso2"))
if unknown:
    print(f"in `article` but not in COUNTRY_ROSTER, so unscorable: {unknown}")

rng = random.Random(SEED)

if PIN:
    ISO2, AS_OF = PIN[0], datetime.date.fromisoformat(PIN[1])
    items = snapshot_select.select(ISO2, AS_OF)
else:
    # Uniform over countries, not over articles: weighting by volume would make
    # this the US notebook, since the US is over half the corpus.
    for attempt in range(1, 11):
        row = rng.choice(known)
        ISO2 = row["country_iso2"]
        AS_OF = rng.choice(pilot_score.anchors(row["from"], row["last"]))
        items = snapshot_select.select(ISO2, AS_OF)
        if len(items) >= MIN_ARTICLES:
            break
        print(f"  draw {attempt}: {ISO2} {AS_OF} returned {len(items)} article(s), "
              f"under MIN_ARTICLES - redrawing")
    else:
        raise RuntimeError(
            f"ten draws in a row came back under {MIN_ARTICLES} articles. That is "
            f"a corpus with holes in it, not bad luck - check `store.counts_by_month()`.")

NAME = config.country_name(ISO2)
WINDOW_START, WINDOW_END = snapshot_select.window(AS_OF)

# The stable ids the model cites back, assigned exactly where
# `pipeline._process_country` assigns them.
for i, it in enumerate(items, start=1):
    it["id"] = f"a{i}"

print(f"\n{'PINNED' if PIN else 'DREW'}  {NAME} ({ISO2})  as of {AS_OF:%Y-%m-%d} "
      f"({AS_OF:%A})")
print(f"  window   [{WINDOW_START:%Y-%m-%d}, {WINDOW_END:%Y-%m-%d})   upper bound strict")
print(f"  articles {len(items)} selected")
print(f"  mode     {SCORING_MODE}")

INFO    backend.data_upsert.data_push: database: dev -> ep-winter-silence-ay9jls7l-pooler.c-5.us-east-2.aws.neon.tech/neondb
6 country/ies with a corpus:


,name,first,last,n
country_iso2,,,,
BR,Brazil,2016-08-01,2026-07-25,3172
ID,Indonesia,2019-01-01,2019-12-31,432
KR,South Korea,2016-08-02,2026-08-01,8566
PT,Portugal,2016-08-01,2026-08-27,9131
TR,Turkey,2016-08-01,2026-08-27,15541
US,United States,2016-08-01,2026-08-01,43643


INFO    backend.news_fetching.snapshot_select: [ID 2019-05-27] 5 of 38 cleared the relevance threshold; topping up to 20 by rank

DREW  Indonesia (ID)  as of 2019-05-27 (Monday)
  window   [2019-04-27, 2019-05-27)   upper bound strict
  articles 20 selected
  mode     masked


---
# A — The data footprint

Three sources feed one snapshot: the articles in the 30-day window, the numeric
series in `indicator_series`, and a handful of static curated files. Nothing else
reaches the model.

### A1 — The articles, and which bodies were refused

`store.read_window` is the whole pool; `snapshot_select.select` is what survived
scoring, the relevance threshold, the abstract-tier cap and the per-theme floor.

The `body_kept` column is the no-future rule. A body captured *on or after* the
anchor is a later edition of the page — publishers edit, append and re-headline —
so it is dropped and the article reaches the scorer as title and abstract.
Thinner evidence, honestly thin, rather than richer evidence that is a lie.
Nothing here drops an *article*.

In [3]:
rows = store.read_window(ISO2, WINDOW_START, WINDOW_END)
selected_ids = {it.get("url") for it in items}

verdicts = []
for row in rows:
    vintage = row.get("body_vintage")
    captured = snapshot_select.capture_date(vintage)
    kept = snapshot_select.usable_body(row, AS_OF)
    if not row.get("body"):
        why = "never had one"
    elif kept:
        why = ("came with the article" if vintage == "api-native"
               else "leakage scan cleared it" if vintage == "live-refetch"
               else f"captured {captured} < anchor")
    else:
        why = (f"captured {captured} >= anchor - a later edition of the page"
               if captured else f"unreadable vintage {vintage!r}")
    verdicts.append({
        "published": row["published_at"].date(),
        "source": row.get("source_system"), "tier": row.get("tier"),
        "body_chars": len(row.get("body") or ""), "body_kept": bool(kept),
        "selected": row["url"] in selected_ids, "why": why,
        "title": (row["title"] or "")[:50],
    })

refused = sum(1 for v in verdicts if v["body_chars"] and not v["body_kept"])
print(f"{len(rows)} article(s) in the window -> {len(items)} selected   "
      f"({refused} body/ies refused as hindsight)")
display(pd.DataFrame(verdicts).sort_values(["selected", "published"],
                                           ascending=[False, False]).head(30))

print("what the scorer got, by relevance:")
display(pd.DataFrame([
    {"id": it["id"], "relevance": round(it.get("relevance_score", 0), 3),
     "theme": it.get("_theme"), "source": it.get("source"),
     "body_chars": len(it.get("text") or ""), "title": it.get("title")}
    for it in items
]).set_index("id"))

38 article(s) in the window -> 20 selected   (0 body/ies refused as hindsight)


,published,source,tier,body_chars,body_kept,selected,why,title
0,2019-05-26,guardian,full,7254,True,False,came with the article,"Ashburn SW7, South Kensington: ‘A dirty stain on i"
1,2019-05-25,guardian,full,2435,True,False,came with the article,Indonesia election: presidential rival challenges
2,2019-05-23,guardian,full,3689,True,False,came with the article,Jakarta riots: Indonesian president says he will n
3,2019-05-22,guardian,full,3891,True,False,came with the article,Indonesia riots: six dead after protesters clash w
4,2019-05-21,guardian,full,2212,True,False,came with the article,Boeing dismissed chance of 'bird strike' that may
5,2019-05-21,guardian,full,3014,True,False,came with the article,Indonesian troops on alert as Widodo wins more tha
6,2019-05-21,guardian,full,8340,True,False,came with the article,Changing travel: the sustainable startups boosted
7,2019-05-16,guardian,full,5203,True,False,came with the article,'Moral panic' over gender identity film sparks bac
8,2019-05-15,guardian,full,7466,True,False,came with the article,"Morning mail: Labor still leads, Ardern deal on ex"
9,2019-05-15,guardian,full,4523,True,False,came with the article,Leaders and tech firms pledge to tackle extremist


what the scorer got, by relevance:


,relevance,theme,source,body_chars,title
id,,,,,
a1,0.98,friction,guardian,3891,Indonesia riots: six dead after protesters clash with tr...
a2,0.90,friction,guardian,2435,Indonesia election: presidential rival challenges Widodo...
a3,0.75,friction,guardian,3689,Jakarta riots: Indonesian president says he will not tol...
a4,0.60,friction,guardian,3014,Indonesian troops on alert as Widodo wins more than half...
a5,0.53,friction,guardian,4632,‘It opened my eyes’: the Indonesian woman fighting for W...
a6,0.10,order,guardian,7254,"Ashburn SW7, South Kensington: ‘A dirty stain on its pos..."
a7,0.10,order,guardian,2212,Boeing dismissed chance of 'bird strike' that may have c...
a8,0.10,order,guardian,8340,Changing travel: the sustainable startups boosted by awa...
a9,0.10,friction,guardian,7466,"Morning mail: Labor still leads, Ardern deal on extremis..."


### A2 — The numbers

Every number the model ever sees comes out of one table, `indicator_series` —
World Bank annuals, IMF monthlies, BIS policy rates and FX, and the curated CSV
drop, all in the same row shape.

Note what is **not** built here. `prepare_llm_payload_pretty` produces the wide
annual panel, and the scorer never sees it: it exists so `upsert_snapshot` can
write the tables the front-end's indicator pane queries, and its
`_meta.generated_at` is where a live run gets its `as_of`. This notebook chose
`AS_OF` itself, so there is nothing for it to do and a World Bank round-trip is
skipped.

In [4]:
def safe(read, what):
    """The pipeline's own per-store resilience: a failed read is absent, not fatal."""
    try:
        return read()
    except Exception as exc:
        print(f"  {what:<20} unavailable ({type(exc).__name__}) - degrading to absent")
        return None


series = safe(lambda: data_push.read_indicator_series(ISO2), "indicator_series") or {}
structural = safe(curated_loader.load_structural_facts, "structural_facts.yaml") or {}

print(f"{len(series)} indicator code(s) with rows for {ISO2}, "
      f"{sum(len(v) for v in series.values()):,} observations in total\n")

display(pd.DataFrame([
    {"code": code_, "label": constants.INDICATOR_REGISTRY.get(code_, {}).get("label"),
     "ledger": constants.INDICATOR_REGISTRY.get(code_, {}).get("ledger"),
     "rows": len(rows_), "first": rows_[0].get("period"), "last": rows_[-1].get("period")}
    for code_, rows_ in sorted(series.items()) if rows_
]).set_index("code"))

print(f"structural facts: {'present' if structural.get(ISO2) else 'ABSENT - '
      'treated as unknown, never guessed'}")
print(f"fx regime:  {constants.FX_REGIMES.get(ISO2)}")
print(f"elections:  {constants.ELECTIONS.get(ISO2, [])}")

INFO    backend.data_fetching.curated_loader: [structural] 5 of 48 countries have a structural block
24 indicator code(s) with rows for ID, 4,159 observations in total



,label,ledger,rows,first,last
code,,,,,
BIS.FX.USD,Exchange rate vs USD,uncertainty,132,2015-08,2026-07
BIS.POLICY.RATE,Policy rate (%),uncertainty,132,2015-07,2026-06
BX.KLT.DINV.WD.GD.ZS,FDI inflow (% GDP),uncertainty,56,1970,2025
CPI.YOY,Inflation (% y/y),uncertainty,897,1960,2026-06
DT.DOD.DSTC.IR.ZS,Short-term external debt (% reserves),uncertainty,11,2014,2024
FM.LBL.BMNY.ZG,Broad money growth (% y/y),friction,12,2014,2025
GC.XPN.INTP.RV.ZS,Interest payments (% revenue),friction,33,1972,2009
GOV_WGI_GE.EST,Government effectiveness (z-score),friction,11,2014,2024
GOV_WGI_PV.EST,Political stability (z-score),uncertainty,26,1996,2024


structural facts: ABSENT - treated as unknown, never guessed
fx regime:  None
elections:  []


### A3 — The evidence payload, which is the thing the model scores

`build_evidence_payload` resolves every registry indicator, buckets each into its
ledger, and adds a `computed` block of derived metrics from
`backend/util/metrics.py`.

It is pure — every store is passed **in** rather than read inside, which is what
makes a historical anchor re-runnable at all: the caller decides which snapshot
of the world it sees. `vintage_as_of=AS_OF` is that decision, and it is the whole
difference between a backfill and a live run. An indicator with no observation is
omitted entirely; absent means absent, never zero.

In [5]:
evidence = llm_payload.build_evidence_payload(
    ISO2,
    as_of=AS_OF,
    series=series,
    fx_regimes=constants.FX_REGIMES,
    elections=constants.ELECTIONS,
    # Withheld entirely for the no-structural diagnostic arm, which is the one
    # thing that separates "the structural facts worked" from "identity never
    # mattered".
    structural={} if SCORING_MODE == "masked_nostructural" else structural,
    vintage_as_of=AS_OF,   # historical. A live run passes None.
)

for section in ("friction_inputs", "uncertainty_inputs",
                "information_inputs", "edge_inputs"):
    keys = list(evidence.get(section, {}))
    print(f"{section:<22} {len(keys):>2}  {', '.join(keys[:4])}{' ...' if len(keys) > 4 else ''}")

print(f"\ncomputed ({len(evidence.get('computed', {}))} metrics from backend/util/metrics.py):")
print(json.dumps(evidence.get("computed"), indent=2, default=str))
print(f"\nstructural block: {'present' if evidence.get('structural') else 'absent'}")
print(f"blocks in the payload: {[k for k in evidence if k != '_meta']}")

INFO    backend.llm.payload: [ID] evidence payload ~1587 tokens (budget 2800)
friction_inputs         9  Government effectiveness (z-score), Political corruption index (0–1, higher = more corrupt), Interest payments (% revenue), Income inequality (Gini) ...
uncertainty_inputs     12  Real GDP growth (% y/y), Current account balance (% GDP), Inflation (% y/y), Political stability (z-score) ...
information_inputs      1  Statistical performance (0–100)
edge_inputs             3  New business density (per 1,000 working-age), Government education spending (% GDP), Human Capital Index (0–1)

computed (6 metrics from backend/util/metrics.py):
{
  "conversion_loss": 0.5993,
  "monetary_dilution": 4.1162,
  "real_policy_rate": 2.802,
  "fx_volatility_24m": 1.448814,
  "precommitted_share": {
    "value": 11.0507,
    "partial": true
  },
  "dependency_trajectory": {
    "current": 9.2836,
    "projected_10y": null,
    "delta": null
  }
}

structural block: absent
blocks in the payload: ['fric

In [6]:
# One ledger unrolled, so the per-value stamps are visible. `staleness_days`
# counts from the end of the period a value describes to AS_OF - how old the
# reading is. `as_of` is a different fact: when it became known to us.
display(pd.DataFrame([
    {"indicator": k, "value": v.get("value"), "period": v.get("period"),
     "freq": v.get("freq"), "stale_days": v.get("staleness_days"),
     "source": v.get("source"), "unit": v.get("unit")}
    for k, v in evidence["friction_inputs"].items() if isinstance(v, dict)
]).set_index("indicator"))

,value,period,freq,stale_days,source,unit
indicator,,,,,,
Government effectiveness (z-score),0.1566,2017,A,512,World Bank WGI,z-score
"Political corruption index (0–1, higher = more corrupt)",0.7300,2018,A,147,World Bank panel,index (0–1)
Interest payments (% revenue),11.0507,2009,A,3434,World Bank panel,% revenue
Income inequality (Gini),36.3000,2018,A,147,World Bank panel,index
Government gross debt (% GDP),29.1960,2018,A,147,IMF WEO 2019-04,% GDP
Government net lending/borrowing (% GDP),-1.7520,2018,A,147,IMF WEO 2019-04,% GDP
Old-age dependency ratio,9.2836,2017,A,512,World Bank WDI,% working-age population
Labour-force participation (% 15+),66.6940,2017,A,512,World Bank WDI,%
Broad money growth (% y/y),8.2755,2017,A,512,World Bank WDI,% y/y


### A4 — What is missing, and why

The other half of a footprint. `payload_health` checks every indicator the
registry promises against what the payload actually carries, and classifies each
absence, because the three reasons want three different fixes:

| | |
|---|---|
| `no row` | nothing was ever fetched — a source problem |
| `vintage bound` | rows exist and every one was published after the anchor — invisible from row counts, and the bug that hid for a whole pilot |
| `unmapped` | it resolved and still did not reach the payload — a wiring problem |

A ledger at **zero** is named rather than left to be inferred from a table.
`fingerprint` is over the delivered indicator set: two rows with the same
`payload_version` and different fingerprints were scored on different evidence
and are not comparable.

In [7]:
health = llm_payload.payload_health(evidence, series, AS_OF, items)
ind = health["indicators"]

print(f"{ind['resolved']}/{ind['expected']} registry indicators reached the payload")
print(f"fingerprint {ind['fingerprint']}")
if ind["empty_ledgers"]:
    print(f"EMPTY LEDGERS: {ind['empty_ledgers']}  <- this is the shape of the vintage bug")

display(pd.DataFrame([
    {"ledger": led, **counts,
     "missing": counts["expected"] - counts["resolved"]}
    for led, counts in ind["by_ledger"].items()
]).set_index("ledger"))

if ind["dropped"]:
    print(f"{len(ind['dropped'])} indicator(s) did not make it:")
    display(pd.DataFrame([
        {"code": c, "reason": why,
         "label": constants.INDICATOR_REGISTRY.get(c, {}).get("label"),
         "rows_in_db": len(series.get(c) or [])}
        for c, why in ind["dropped"].items()
    ]).set_index("code"))

print(f"\narticles: {json.dumps(health.get('articles'), indent=2, default=str)}")

24/38 registry indicators reached the payload
fingerprint 5f17a77b93c2472a


,expected,resolved,missing
ledger,,,
friction,14,9,5
uncertainty,16,11,5
information,4,1,3
edge,4,3,1


14 indicator(s) did not make it:


,reason,label,rows_in_db
code,,,
GC.TAX.TOTL.GD.ZS,no row,Tax revenue (% GDP),0
GOV.DEBT.DOMESTIC.SHARE,no row,Government debt held by residents (% of total),0
GOV.DEBT.FX.SHARE,no row,Government debt in foreign currency (% of total),0
INFORMAL.PCT.GDP,no row,Informal economy (% GDP),0
NIIP.GDP,no row,Net international investment position (% GDP),0
OBS.SCORE,no row,Open Budget Survey (0–100),0
OECD.PISA.MEAN,no row,PISA mean score (math/reading/science),0
OECD.TAX.WEDGE,no row,Labour tax wedge (% labour cost),0
RESERVES.USD,no row,Total reserves (USD),0



articles: {
  "articles": 20,
  "by_theme": {
    "friction": 10,
    "order": 7,
    "security": 2,
    "information": 1,
    "edge": 0,
    "broad": 0
  },
  "thin_themes": [
    "broad",
    "edge",
    "information"
  ],
  "theme_floor": 2,
  "by_tier": {
    "full": 20
  },
  "with_body": 20,
  "clipped_at_max": 0
}


### A5 — Masking, then the digests

Two passes of masking, and stage one in between them.

**Layer 1** is the gazetteer: a hand-written list, offline and deterministic,
replacing country names, cities, people, parties, currencies and institutions
with the roles they play — "the country", "the capital", "the central bank".
Every number is untouched. It runs *before* the digests rather than at the model
call, because a digest written from named text carries the name into the prompt
however clean the article beside it is.

**Stage one** turns each article body into a small structured digest with the
cheap model. That is what keeps the prompt bounded: twenty full bodies would be
tens of thousands of tokens, and the scorer reads digests instead. Cached on a
hash of the content, so a re-run of the same anchor costs nothing.

**Layer 2** is a model pass over the two or three bodies the scorer reads end to
end. The gazetteer does not know this week's finance minister, and a full body
names one far more often than it names the country. It **fails closed**: a
rewrite that errors or comes back empty leaves the article with no body at all.

In [8]:
scored = rewrite.mask_items(items, ISO2) if MASKED else items
DISPLAY = langchain_llm.MASKED_COUNTRY_LABEL if MASKED else NAME

demo = next((i for i, it in enumerate(items)
             if gazetteer.mentions(it.get("title") or "", ISO2)), 0)
print(f"country label in the prompt: {DISPLAY!r}\n")
print(f"BEFORE  {items[demo]['title']}")
print(f"AFTER   {scored[demo]['title']}")

with usage.meter() as meter:
    scored = digest_engine.digest_articles(
        scored, country_display=DISPLAY, iso2=ISO2, as_of=AS_OF,
        # About what the digest model *writes*, not what it reads - the text is
        # masked already. Without it, "actors: who did what to whom" reads as an
        # instruction to name people, and people are what a list cannot know.
        masked=MASKED,
        # Keyed on content rather than on as_of, so the four consecutive weekly
        # anchors that share an article pay for its digest once.
        content_cache=store,
    )
    fulltext_ids = digest_engine.select_fulltext_ids(scored)

    was = {it["id"]: len(digest_engine.article_input_text(it))
           for it in scored if it["id"] in fulltext_ids}
    if MASKED:
        pipeline._rewrite_fulltext(scored, fulltext_ids, ISO2, cache=store)

digested = [it for it in scored if isinstance(it.get("digest"), dict)]
print(f"\n{len(digested)}/{len(scored)} digested   spend ${meter.spend_usd:.4f}")
print(f"full text goes to the scorer for: {fulltext_ids}")
for it in scored:
    if it["id"] in fulltext_ids:
        now = len(digest_engine.article_input_text(it))
        print(f"  {it['id']}  {was[it['id']]:>6,} -> {now:>6,} chars   "
              f"{'title-only (failed closed)' if now == 0 else 'rewritten'}")

display(pd.DataFrame([
    {"id": it["id"], "severity": it.get("stage1_severity"),
     "full_text": it["id"] in fulltext_ids,
     "what_happened": (it.get("digest") or {}).get("what_happened")}
    for it in scored
]).sort_values("severity", ascending=False).set_index("id"))

country label in the prompt: 'the country'

BEFORE  Indonesia riots: six dead after protesters clash with troops over election result
AFTER   the country riots: six dead after protesters clash with troops over election result
INFO    backend.llm.digest_engine: [ID] digests: ok=20 cached=0 content-cached=0 failed=0 swept=20
INFO    backend.util.pipeline: [ID] full-text rewrites: 0 cached, 3 fresh

20/20 digested   spend $0.0114
full text goes to the scorer for: ['a1', 'a2', 'a3']
  a1   3,928 ->  3,885 chars   rewritten
  a2   2,454 ->  2,474 chars   rewritten
  a3   3,720 ->  3,683 chars   rewritten


,severity,full_text,what_happened
id,,,
a1,85.0,True,Six people died and more than 200 were injured in the ca...
a2,60.0,True,The defeated candidate in the country’s presidential ele...
a3,60.0,True,Political tension over disputed election results led to ...
a4,60.0,False,Heavily armed troops are on high alert amid fears of civ...
a5,60.0,False,A human rights lawyer is taking the police to court over...
a12,60.0,False,A governing party-controlled state senate has passed a b...
a18,60.0,False,the capital announced it will partially withdraw from th...
a10,40.0,False,Global leaders and heads of large technology companies p...
a11,40.0,False,Pilots from a foreign airline confronted a large domesti...


In [9]:
# One digest in full - the highest-severity article, which is also the one whose
# body the scorer is most likely to read end to end.
focus = max(digested, key=lambda it: it.get("stage1_severity") or 0.0)
print(f"{focus['id']}  severity {focus.get('stage1_severity')}   "
      f"{len(digest_engine.article_input_text(focus)):,} chars in, "
      f"{len(json.dumps(focus['digest'])):,} out\n")
print(json.dumps(focus["digest"], indent=2, ensure_ascii=False))

a1  severity 85.0   3,885 chars in, 497 out

{
  "what_happened": "Six people died and more than 200 were injured in the capital after protesters clashed with security forces.",
  "actors": "Supporters of an unsuccessful presidential candidate clashed with security forces, resulting in deaths and injuries.",
  "numbers": "6, 200, 55.5%, 20, 30,000, 2014, 46",
  "transmission": "not stated",
  "directly_about_country": true,
  "stage1_severity": 85,
  "masked_title": "the country riots: six dead after protesters clash with troops over election result"
}


---
# B — Everything the model sees

One `SystemMessage`. No user turn, no conversation, no tool definitions.

`langchain_llm.build_prompt` is the function `country_llm_score` itself calls to
render the prompt, so what prints below is not a reconstruction of the prompt —
it is the prompt. (`test_llm.TestTheBuilderIsWhatIsSent` asserts the two are byte
identical, on both the masked and the named arm.)

Masking happens again on the way out, because the evidence payload names the
country in its `_meta` and its series labels and the digests were generated after
layer 1. Then `assert_clean` scans the blocks that are about to be sent and
**raises** if any roster country's name survived. Not the whole prompt: the
template's own worked examples name Australia and China, and those are
instructions rather than evidence about anyone.

In [10]:
built = langchain_llm.build_prompt(
    country_display=NAME,          # masked to "the country" inside the call
    payload=evidence,
    articles=scored,
    as_of=AS_OF,
    fulltext_ids=fulltext_ids,
    mask_iso2=ISO2 if MASKED else None,
)
PROMPT = built["prompt"]

print("mask integrity: clean - assert_clean found no roster country name "
      "in anything about to be sent" if MASKED else "named arm - no gate")
print(f"prompt_version stamped: {built['prompt_version']}")
print(f"appended rule blocks:   {len(built['rules']):,} chars"
      f"{' (none - base template)' if not built['rules'] else ''}\n")

print(f"country       {built['country_display']!r}")
print(f"as_of_date    {AS_OF.isoformat()}   (treated as 'today' by the prompt)\n")
for label, key in (("EVIDENCE_JSON", "evidence_json"),
                   ("ARTICLES_JSON", "articles_json"),
                   ("FULL_TEXT", "fulltext_block")):
    blob = built[key]
    print(f"{label:<16}{len(blob):>9,} chars  ~{len(blob)//4:>6,} tokens")
print(f"{'TOTAL PROMPT':<16}{len(PROMPT):>9,} chars  ~{len(PROMPT)//4:>6,} tokens")
print(f"\nfor comparison, every article body in full: "
      f"{sum(len(digest_engine.article_input_text(it)) for it in built['articles']):,} "
      f"chars - what the digests replaced")

print("\n" + "-" * 78 + "\nARTICLES_JSON, first entry:\n")
print(json.dumps(json.loads(built["articles_json"])[0], indent=2, ensure_ascii=False))

mask integrity: clean - assert_clean found no roster country name in anything about to be sent
prompt_version stamped: v4.5-no-publisher
appended rule blocks:   0 chars (none - base template)

country       'the country'
as_of_date    2019-05-27   (treated as 'today' by the prompt)

EVIDENCE_JSON       6,383 chars  ~ 1,595 tokens
ARTICLES_JSON      17,749 chars  ~ 4,437 tokens
FULL_TEXT          10,347 chars  ~ 2,586 tokens
TOTAL PROMPT       46,781 chars  ~11,695 tokens

for comparison, every article body in full: 96,753 chars - what the digests replaced

------------------------------------------------------------------------------
ARTICLES_JSON, first entry:

{
  "id": "a1",
  "published_at": "2019-05-22",
  "title": "the country riots: six dead after protesters clash with troops over election result",
  "digest": {
    "what_happened": "Six people died and more than 200 were injured in the capital after protesters clashed with security forces.",
    "actors": "Supporters of an unsu

### The prompt, verbatim

This is the entire string the model receives. Nothing else is sent.

In [11]:
print("=" * 78)
print(PROMPT)
print("=" * 78)

You are a senior sovereign risk analyst. Assess investor risk for the country
as of 2019-05-27, using ONLY the evidence below.
Treat 2019-05-27 as today: this evidence is your complete knowledge of the
world. Do not use anything you know about events after this date.

Every value in EVIDENCE_JSON carries `as_of` and `staleness_days` — the date it
became known and how old it is on 2019-05-27. Weigh a fresh reading more than
a stale one, and say so when a stale one is carrying an argument. A missing
indicator is absent from the evidence entirely; treat absence as absence, never
as zero and never as reassurance.

# --- The country is not named, deliberately ---
This evidence describes a real country whose identity has been withheld from
you. Country names, cities, people, parties, currencies and institutions have
been replaced by the roles they play: "the country", "the capital", "the central
bank", "the finance minister", "the local currency". Every NUMBER is untouched —
inflation prints

### What it is asked to return, and how

The schema is bound with `strict=True`, so the model cannot return a field the
schema does not name. `strict` enforces *shape*, not `minimum`/`maximum` — which
is why `_from_100` clamps on the way back.

In [12]:
schema = langchain_llm._SCHEMA_BY_PROMPT_VARIANT.get(
    provenance.prompt_variant(), ai_constants.RISK_SCHEMA_V3)

print(f"model         {ai_client.scoring_model()}")
print(f"temperature   0.0     seed {ai_client.SEED}     max_retries 0")
print(f"messages      [SystemMessage(content=<the prompt above>)]   - no user turn")
print(f"tools         none")
print(f"schema        {provenance.prompt_variant() or 'RISK_SCHEMA_V3'}, strict=True\n")
print(json.dumps(schema, indent=2)[:4000])

model         gpt-4o-2024-08-06
temperature   0.0     seed 42     max_retries 0
messages      [SystemMessage(content=<the prompt above>)]   - no user turn
tools         none
schema        RISK_SCHEMA_V3, strict=True

{
  "title": "CountryRiskAssessmentV3",
  "description": "The model's judgement under the friction framework: condition flags as observations, four ledger scores with the evidence behind the three risk-bearing ones, per-article impacts with topic grouping, two horizons, and a short summary. All scores are integers 0-100. No code downstream alters any score.",
  "type": "object",
  "properties": {
    "condition_flags": {
      "title": "ConditionFlags",
      "type": "object",
      "properties": {
        "war_on_territory": {
          "type": "boolean"
        },
        "internal_conflict_level": {
          "type": "string",
          "enum": [
            "none",
            "A",
            "B",
            "C"
          ]
        },
        "emergency_rule": {
    

---
# C — Send it

**Off by default.** Set `RUN_SCORER = True` and re-run this cell — everything
above is already in memory, so it is one call and about two cents.

One call, and **nothing edits the score afterwards**. There were floors, a cap
and a sanctions gate; they are gone. The single assignment to `score` in the
whole backend is the model's own `score_12m`, rescaled to 0-1.

In [13]:
RUN_SCORER = False   # True = one real scoring call, ~$0.02

if RUN_SCORER:
    with usage.meter() as scoring_meter:
        llm_output = langchain_llm.country_llm_score(
            country_display=NAME,
            payload=evidence,
            articles=scored,
            as_of=AS_OF,
            fulltext_ids=fulltext_ids,
            mask_iso2=ISO2 if MASKED else None,
        )
    # A failed call returns the no-score shape rather than raising, so nothing
    # downstream would notice on its own.
    assert llm_output["score"] is not None, "scoring call failed - see the log above"
    print(f"{NAME} ({ISO2})  {AS_OF}   spend ${scoring_meter.spend_usd:.4f}\n")
    print(f"risk  {llm_output['score']:.2f} (12m)   {llm_output['score_3m']:.2f} (3m)"
          f"{'   RESTRICTED' if llm_output['non_investable'] else ''}\n")
    print(llm_output["bullet_summary"])
else:
    llm_output = None
    print("RUN_SCORER is False - flip it and re-run this cell to spend ~$0.02")

RUN_SCORER is False - flip it and re-run this cell to spend ~$0.02


### The raw return, whole

In [14]:
if llm_output:
    print(json.dumps(llm_output, indent=2, ensure_ascii=False, default=str))
else:
    print("nothing to show - RUN_SCORER is False")

nothing to show - RUN_SCORER is False


### Its reasoning, in its own citations

`subscore_evidence` is the model naming which indicators and which article ids
moved each ledger. `condition_flags` are the binary readings it was asked for.
`news_article_scores` is the per-article impact that drives Top-3 selection on
the dashboard.

The prompt asks for **integers 0-100**, because that grid has the rank resolution
the roster needs; everything downstream speaks 0-1. `_from_100` runs the moment
the call returns, so a 0-100 number never leaves `langchain_llm.py`.

In [15]:
if llm_output:
    display(pd.DataFrame([
        {"ledger": k, "0-1": v, "0-100 (as the model said it)": pipeline._to_100(v)}
        for k, v in (llm_output.get("ledger_scores") or {}).items()
    ]).set_index("ledger"))

    print("condition_flags:")
    for k, v in (llm_output.get("condition_flags") or {}).items():
        print(f"  {k:<28} {v}")

    print("\nsubscore_evidence:")
    for ledger, cites in (llm_output.get("subscore_evidence") or {}).items():
        print(f"  {ledger}: {cites}")

    print(f"\nevidence_coverage  {llm_output['evidence_coverage']}")
    print(f"legal_gate         {llm_output.get('legal_gate')}")
    print(f"schema_violations  {llm_output.get('schema_violations') or 'none'}")
    print(f"stamps             {llm_output['model_id']}  "
          f"prompt {llm_output['prompt_version']}  "
          f"policy {llm_output['policy_version']}")

    # Ranked off the *unmasked* items: these are shown to a human who knows
    # perfectly well which country they are reading about.
    by_id = {it["id"]: it for it in items}
    display(pd.DataFrame([
        {"id": a["id"], "impact": a.get("impact"), "topic": a.get("topic_group"),
         "title": (by_id.get(a["id"]) or {}).get("title")}
        for a in (llm_output.get("news_article_scores") or [])
    ]).sort_values("impact", ascending=False).set_index("id"))
else:
    print("nothing to show - RUN_SCORER is False")

nothing to show - RUN_SCORER is False


## Where this stops

The next line in `pipeline._process_country` is `data_push.upsert_snapshot(...)`,
which writes `risk_snapshot`, `risk_snapshot_article`, `indicator` and
`yearly_value`, and after it `store.write_run(...)` records the ledger row a
resume reads. This notebook calls neither, so nothing on the dashboard moved and
the backfill does not think this week is done.

The real thing is the CLI, and every command that spends money prints its
projection and waits for a yes:

```
python -m backend.util.pilot.run score --country PT --since 2019-01-01 --until 2019-12-31
```

In [16]:
print(f"{NAME} ({ISO2})   as_of {AS_OF}   mode {SCORING_MODE}")
print(f"  window        [{WINDOW_START:%Y-%m-%d}, {WINDOW_END:%Y-%m-%d})   "
      f"{len(rows)} in, {len(items)} selected, {refused} body/ies refused")
print(f"  evidence      {health['indicators']['resolved']}/"
      f"{health['indicators']['expected']} indicators   "
      f"fingerprint {health['indicators']['fingerprint']}")
print(f"  digests       {len(digested)}/{len(scored)}   full text {fulltext_ids}")
print(f"  prompt        {len(PROMPT):,} chars   version {built['prompt_version']}")
print(f"  score         {f'{llm_output["score"]:.2f}' if llm_output else 'not requested'}")
print(f"  wrote         llm_artifact cache rows only "
      f"(RISK_DB_TARGET={os.getenv('RISK_DB_TARGET')})")

Indonesia (ID)   as_of 2019-05-27   mode masked
  window        [2019-04-27, 2019-05-27)   38 in, 20 selected, 0 body/ies refused
  evidence      24/38 indicators   fingerprint 5f17a77b93c2472a
  digests       20/20   full text ['a1', 'a2', 'a3']
  prompt        46,781 chars   version v4.5-no-publisher
  score         not requested
  wrote         llm_artifact cache rows only (RISK_DB_TARGET=dev)
